# Sector 2 — Stage 1: Q-CHAT-10 ASD Detection — Reduced Feature Model

**Improvement over Stage 0:** Drop administrative `Who_completed_the_test_*` columns (3 features) that introduce response-style bias and are absent at deployment.  
**Features used:** A1–A10 (behavioural), Sex_M, Family_mem_with_ASD_Yes → **12 features**  
**SHAP fix:** Use `shap.Explainer` (XGBoost 2.x compatible) instead of the deprecated `shap.TreeExplainer`.

**Dataset:** `pre_processed_data_combined.csv` — unified Q-CHAT-10 dataset (7 530 rows)

**Pipeline:**
1. Data Preparation (drop Who cols)
2. Train 4 Classifiers with 5-Fold Stratified CV
3. Best Model Selection
4. Calibration Check (XGBoost)
5. Save Trained Model
6. SHAP Analysis (fixed API)
7. Extract and Save Probabilities
8. Stage 0 vs Stage 1 Comparison

## Section 1 — Data Preparation

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded successfully.')

In [ ]:
DATA_PATH = r"C:\Users\Yasindu\Desktop\Stuff\Research\Datasets\PrePROCESSED DATA\pre_processed_data_combined.csv"

data = pd.read_csv(DATA_PATH)
print(f"Dataset shape (raw): {data.shape}")
print(f"Columns: {data.columns.tolist()}")
data.head()

In [ ]:
# Drop administrative columns — these encode who filled the form, not child behaviour.
# At deployment (mobile app) the parent always fills the form, so these columns
# introduce selection bias and will be near-constant at inference time.
WHO_COLS = [
    'Who_completed_the_test_Health Care Professional',
    'Who_completed_the_test_Others',
    'Who_completed_the_test_School and NGO',
]

cols_present = [c for c in WHO_COLS if c in data.columns]
data = data.drop(columns=cols_present)

print(f"Dropped {len(cols_present)} Who_completed_the_test_* columns.")
print(f"Dataset shape (after drop): {data.shape}")
print(f"\nRemaining columns ({len(data.columns)}):")
for col in data.columns:
    print(f"  {col} — {data[col].dtype}")

In [ ]:
# Convert all bool columns to int (0/1) for scikit-learn / XGBoost compatibility
bool_cols = data.select_dtypes(include='bool').columns.tolist()
data[bool_cols] = data[bool_cols].astype(int)

print(f"Converted {len(bool_cols)} bool → int columns.")
print("\nDtype check after conversion:")
print(data.dtypes)

In [ ]:
TARGET = 'ASD_traits_Yes'

X = data.drop(columns=[TARGET])
y = data[TARGET]

print(f"Feature matrix X : {X.shape}")
print(f"Target vector  y : {y.shape}")
print(f"\nFeatures ({X.shape[1]}):")
for col in X.columns:
    print(f"  {col}")

print(f"\nOverall class distribution:")
vc = y.value_counts()
print(f"  Non-ASD (0): {vc.get(0, 0)}  ({vc.get(0, 0)/len(y)*100:.1f}%)")
print(f"  ASD     (1): {vc.get(1, 0)}  ({vc.get(1, 0)/len(y)*100:.1f}%)")

In [ ]:
# Stratified 80/20 split — identical seed to Stage 0 for a fair comparison
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=" * 50)
print("Train / Test Split (stratified, random_state=42)")
print("=" * 50)
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")

print(f"\nClass balance — TRAIN set ({len(y_train)} rows):")
vc_tr = y_train.value_counts()
print(f"  Non-ASD (0): {vc_tr.get(0,0)}  ({vc_tr.get(0,0)/len(y_train)*100:.1f}%)")
print(f"  ASD     (1): {vc_tr.get(1,0)}  ({vc_tr.get(1,0)/len(y_train)*100:.1f}%)")

print(f"\nClass balance — TEST set ({len(y_test)} rows):")
vc_te = y_test.value_counts()
print(f"  Non-ASD (0): {vc_te.get(0,0)}  ({vc_te.get(0,0)/len(y_test)*100:.1f}%)")
print(f"  ASD     (1): {vc_te.get(1,0)}  ({vc_te.get(1,0)/len(y_test)*100:.1f}%)")

## Section 2 — Train 4 Classifiers with 5-Fold Stratified CV

In [ ]:
# Create output directories
os.makedirs('results', exist_ok=True)

# Define the four classifiers — identical hyperparameters to Stage 0
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=100, eval_metric='logloss',
                                         random_state=42, verbosity=0),
    'SVM':                 SVC(kernel='linear', probability=True, random_state=42),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

print('Models and CV strategy defined.')
print('StratifiedKFold: n_splits=5, shuffle=True, random_state=42')

In [ ]:
cv_records = {}

for name, model in models.items():
    print(f'Training {name} (5-fold CV)...')
    scores = cross_validate(
        model, X_train, y_train,
        cv=skf, scoring=scoring, return_train_score=False, n_jobs=-1
    )
    cv_records[name] = {
        'Accuracy Mean':   round(scores['test_accuracy'].mean(),  4),
        'Accuracy Std':    round(scores['test_accuracy'].std(),   4),
        'Precision Mean':  round(scores['test_precision'].mean(), 4),
        'Precision Std':   round(scores['test_precision'].std(),  4),
        'Recall Mean':     round(scores['test_recall'].mean(),    4),
        'Recall Std':      round(scores['test_recall'].std(),     4),
        'F1 Mean':         round(scores['test_f1'].mean(),        4),
        'F1 Std':          round(scores['test_f1'].std(),         4),
        'ROC-AUC Mean':    round(scores['test_roc_auc'].mean(),   4),
        'ROC-AUC Std':     round(scores['test_roc_auc'].std(),    4),
    }
    print(f'  Accuracy : {scores["test_accuracy"].mean():.4f} ± {scores["test_accuracy"].std():.4f}')
    print(f'  ROC-AUC  : {scores["test_roc_auc"].mean():.4f} ± {scores["test_roc_auc"].std():.4f}')

print('\nAll CV training complete.')

In [ ]:
cv_df = pd.DataFrame(cv_records).T.sort_values('ROC-AUC Mean', ascending=False)

print('=== 5-Fold Cross-Validation Results (sorted by ROC-AUC) ===')
display(cv_df)

cv_df.to_csv('results/cv_results_qchat.csv')
print('\nSaved: results/cv_results_qchat.csv')

## Section 3 — Test Set Evaluation

In [ ]:
# Train each model on full training set, evaluate on held-out test set
test_records = {}
trained_models = {}

for name, model in models.items():
    print(f'Fitting {name} on full train set...')
    model.fit(X_train, y_train)
    trained_models[name] = model

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    test_records[name] = {
        'Accuracy':  round(accuracy_score(y_test, y_pred),  4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred),    4),
        'F1':        round(f1_score(y_test, y_pred),        4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob),   4),
    }

test_df = pd.DataFrame(test_records).T.sort_values('ROC-AUC', ascending=False)

print('\n=== Test Set Results (sorted by ROC-AUC) ===')
display(test_df)

test_df.to_csv('results/test_results_qchat.csv')
print('\nSaved: results/test_results_qchat.csv')

## Section 4 — Best Model Selection

In [ ]:
os.makedirs('plots', exist_ok=True)

best_model_name = test_df.index[0]
best_model      = trained_models[best_model_name]
best_metrics    = test_df.loc[best_model_name]

print('=' * 50)
print(f'Best Model : {best_model_name}')
print('=' * 50)
print(f'  Accuracy  : {best_metrics["Accuracy"]:.4f}')
print(f'  Precision : {best_metrics["Precision"]:.4f}')
print(f'  Recall    : {best_metrics["Recall"]:.4f}')
print(f'  F1 Score  : {best_metrics["F1"]:.4f}')
print(f'  ROC-AUC   : {best_metrics["ROC-AUC"]:.4f}')

In [ ]:
y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Non-ASD (0)', 'ASD (1)'],
    yticklabels=['Non-ASD (0)', 'ASD (1)'],
    annot_kws={'size': 14}, ax=ax
)
ax.set_title(f'Confusion Matrix — {best_model_name}\n(Test set, n={len(y_test)})',
             fontsize=13, pad=10)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('plots/confusion_matrix_qchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/confusion_matrix_qchat.png')

## Section 5 — Calibration Check (XGBoost)

In [ ]:
xgb_model  = trained_models['XGBoost']
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_test_arr = y_test.values

# Reliability diagram data
fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test_arr, y_prob_xgb, n_bins=10
)

# Brier Score
brier_score = float(np.mean((y_prob_xgb - y_test_arr) ** 2))

# Expected Calibration Error (ECE)
def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece  = 0.0
    n    = len(y_true)
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc  = float(y_true[mask].mean())
        bin_conf = float(y_prob[mask].mean())
        ece     += mask.sum() * abs(bin_acc - bin_conf)
    return ece / n

ece = compute_ece(y_test_arr, y_prob_xgb)

print('=== XGBoost Calibration Metrics (test set) ===')
print(f'  Brier Score : {brier_score:.4f}  (lower is better; perfect = 0.0)')
print(f'  ECE         : {ece:.4f}  (lower is better; perfect calibration = 0.0)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ---- Left: Reliability Diagram ----
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Perfect calibration')
axes[0].plot(
    mean_predicted_value, fraction_of_positives,
    's-', color='steelblue', linewidth=1.8, markersize=6,
    label=f'XGBoost  (Brier={brier_score:.3f}, ECE={ece:.3f})'
)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].set_xlabel('Mean Predicted Probability', fontsize=12)
axes[0].set_ylabel('Fraction of Positives (True ASD rate)', fontsize=12)
axes[0].set_title('Reliability Diagram (Calibration Curve)', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# ---- Right: Probability Distribution by True Class ----
axes[1].hist(
    y_prob_xgb[y_test_arr == 0], bins=30,
    alpha=0.6, color='steelblue', label='Non-ASD (true=0)', density=True
)
axes[1].hist(
    y_prob_xgb[y_test_arr == 1], bins=30,
    alpha=0.6, color='tomato', label='ASD (true=1)', density=True
)
axes[1].axvline(
    0.5, color='black', linestyle='--', linewidth=1.4, label='Threshold = 0.5'
)
axes[1].set_xlabel('P(ASD)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Predicted Probability Distribution by True Class', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.suptitle(
    f'XGBoost Calibration Analysis   |   Brier={brier_score:.4f}   ECE={ece:.4f}',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('plots/calibration_qchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/calibration_qchat.png')

## Section 6 — Save Trained Model

In [ ]:
os.makedirs('models', exist_ok=True)

# Train final XGBoost on the FULL dataset (all 7530 rows)
print('Training final XGBoost on full dataset (7530 rows)...')
xgb_final = XGBClassifier(
    n_estimators=100, eval_metric='logloss',
    random_state=42, verbosity=0
)
xgb_final.fit(X, y)
print('Training complete.')

feature_columns = X.columns.tolist()

joblib.dump(xgb_final,       'models/xgboost_qchat_stage1.pkl')
joblib.dump(feature_columns, 'models/qchat_feature_columns.pkl')

model_size = os.path.getsize('models/xgboost_qchat_stage1.pkl')
cols_size  = os.path.getsize('models/qchat_feature_columns.pkl')

print('\n=== Saved Files ===')
print(f'  models/xgboost_qchat_stage1.pkl  ({model_size / 1024:.1f} KB)')
print(f'  models/qchat_feature_columns.pkl ({cols_size  / 1024:.1f} KB)')

print(f'\nFeature columns saved ({len(feature_columns)}):')
for col in feature_columns:
    print(f'  {col}')

## Section 7 — SHAP Analysis

Uses `shap.Explainer` (the unified API) which is compatible with XGBoost 2.x.  
This fixes the `AttributeError: 'XGBTreeModelLoader' object has no attribute 'num_trees'`
that occurred in Stage 0 with the deprecated `shap.TreeExplainer`.

In [ ]:
print('Computing SHAP values (shap.Explainer — XGBoost 2.x compatible)...')

# shap.Explainer auto-selects TreeExplainer for XGBoost using the new API
explainer   = shap.Explainer(xgb_final, X)
shap_values = explainer(X)

print(f'SHAP values shape: {shap_values.values.shape}')

# Top 12 features by mean absolute SHAP
mean_shap = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=feature_columns
).sort_values(ascending=False)

print('\n=== Feature Ranking by Mean |SHAP| ===')
for rank, (feat, val) in enumerate(mean_shap.items(), start=1):
    print(f'  {rank:2d}. {feat:<55s} {val:.4f}')

In [ ]:
# SHAP beeswarm plot
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values.values, X,
    feature_names=feature_columns,
    show=False, plot_size=None
)
plt.title(
    'SHAP Feature Importance — Beeswarm Plot\n(XGBoost Stage 1, 12 features, full dataset)',
    fontsize=13, pad=10
)
plt.tight_layout()
plt.savefig('plots/shap_summary_qchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/shap_summary_qchat.png')

In [ ]:
# SHAP mean absolute importance bar chart
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values.values, X,
    feature_names=feature_columns,
    plot_type='bar',
    show=False, plot_size=None
)
plt.title(
    'SHAP Mean Absolute Feature Importance\n(XGBoost Stage 1, 12 features, full dataset)',
    fontsize=13, pad=10
)
plt.tight_layout()
plt.savefig('plots/shap_bar_qchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/shap_bar_qchat.png')

### DSM-5 Mapping of Q-CHAT-10 Items

Each Q-CHAT-10 question maps to a specific DSM-5 ASD diagnostic criterion:

| Q-CHAT Item | DSM-5 Criterion | Behaviour Assessed |
|---|---|---|
| **Q1 (A1)** | A1 — Social-emotional reciprocity | Response to own name |
| **Q2 (A2)** | A1 — Social-emotional reciprocity | Eye contact quality |
| **Q3 (A3)** | A2 — Nonverbal communication | Protodeclarative pointing |
| **Q4 (A4)** | A2 — Nonverbal communication | Pointing to show shared interest |
| **Q5 (A5)** | A3 — Peer relationships | Pretend / imaginative play |
| **Q6 (A6)** | A2 — Nonverbal communication | Joint attention / gaze following |
| **Q7 (A7)** | A1 — Social-emotional reciprocity | Empathy / comfort response |
| **Q8 (A8)** | B — Restricted / repetitive behaviours | Typicality of first words |
| **Q9 (A9)** | A2 — Nonverbal communication | Use of simple gestures |
| **Q10 (A10)** | B — Restricted / repetitive behaviours | Visual fixation / staring |

**Note on dropped features:**  
`Who_completed_the_test_*` columns were removed because they reflect *respondent type* rather than child behaviour, introducing selection bias without clinical grounding. The remaining features (A1–A10, Sex_M, Family_mem_with_ASD_Yes) are all clinically validated ASD indicators.

## Section 8 — Extract and Save Probabilities

In [ ]:
os.makedirs('data', exist_ok=True)

# Generate P(ASD) for every row using final model (trained on full dataset)
probs = xgb_final.predict_proba(X)[:, 1]
preds = (probs >= 0.5).astype(int)

prob_df = pd.DataFrame({
    'sample_index':    np.arange(len(X)),
    'P_ASD':           probs,
    'predicted_label': preds,
    'true_label':      y.values,
})

prob_df.to_csv('data/qchat_probabilities_stage1.csv', index=False)
print(f'Saved: data/qchat_probabilities_stage1.csv  ({len(prob_df)} rows)')

print('\n--- First 10 rows ---')
display(prob_df.head(10))

print('\n--- Prediction distribution (threshold = 0.5) ---')
pred_vc = prob_df['predicted_label'].value_counts().rename({0: 'Non-ASD (0)', 1: 'ASD (1)'})
print(pred_vc.to_string())

print('\n--- Mean P(ASD) by true class ---')
mean_by_class = (
    prob_df.groupby('true_label')['P_ASD']
    .mean()
    .rename({0: 'Non-ASD (true=0)', 1: 'ASD (true=1)'})
)
print(mean_by_class.to_string())

## Section 9 — Stage 0 vs Stage 1 Comparison

Side-by-side comparison of the 15-feature (Stage 0) and 12-feature (Stage 1) XGBoost models.  
Stage 0 results are hardcoded from the recorded notebook outputs.

In [ ]:
# Stage 0 baseline — hardcoded from Stage_0/qchat_model_training.ipynb outputs
stage0_xgb = {
    'Accuracy':    0.9456,
    'Precision':   0.9168,
    'Recall':      0.9551,
    'F1':          0.9355,
    'ROC-AUC':     0.9806,
    'Brier Score': 0.0448,
    'ECE':         0.0244,
    'Features':    15,
    'CV Acc Mean': 0.9334,
    'CV AUC Mean': 0.9773,
}

# Stage 1 — current run (XGBoost, test set)
xgb_test = test_df.loc['XGBoost']
stage1_xgb = {
    'Accuracy':    xgb_test['Accuracy'],
    'Precision':   xgb_test['Precision'],
    'Recall':      xgb_test['Recall'],
    'F1':          xgb_test['F1'],
    'ROC-AUC':     xgb_test['ROC-AUC'],
    'Brier Score': round(brier_score, 4),
    'ECE':         round(ece, 4),
    'Features':    12,
    'CV Acc Mean': cv_df.loc['XGBoost', 'Accuracy Mean'],
    'CV AUC Mean': cv_df.loc['XGBoost', 'ROC-AUC Mean'],
}

comp_df = pd.DataFrame({'Stage 0 (15 feat)': stage0_xgb, 'Stage 1 (12 feat)': stage1_xgb})
comp_df['Delta (S1 - S0)'] = comp_df['Stage 1 (12 feat)'] - comp_df['Stage 0 (15 feat)']

print('=== XGBoost: Stage 0 (15 features) vs Stage 1 (12 features) ===')
display(comp_df.round(4))

comp_df.to_csv('results/stage0_vs_stage1_comparison.csv')
print('\nSaved: results/stage0_vs_stage1_comparison.csv')

In [ ]:
# Bar chart comparing the five core test-set metrics
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
s0_vals = [stage0_xgb[m] for m in metrics_to_plot]
s1_vals = [stage1_xgb[m] for m in metrics_to_plot]

x       = np.arange(len(metrics_to_plot))
width   = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
bars0 = ax.bar(x - width/2, s0_vals, width, label='Stage 0 (15 features)',
               color='steelblue', alpha=0.85)
bars1 = ax.bar(x + width/2, s1_vals, width, label='Stage 1 (12 features)',
               color='tomato', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, fontsize=12)
ax.set_ylim(0.85, 1.01)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(
    'XGBoost Test Set: Stage 0 (15 feat) vs Stage 1 (12 feat)\n'
    'Who_completed_the_test_* columns removed in Stage 1',
    fontsize=13
)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bar in bars0:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('plots/stage0_vs_stage1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/stage0_vs_stage1_comparison.png')

## Final Summary — Saved Outputs

| File | Description |
|---|---|
| `results/cv_results_qchat.csv` | 5-fold CV metrics (mean ± std) for all 4 classifiers |
| `results/test_results_qchat.csv` | Held-out test set metrics for all 4 classifiers |
| `results/stage0_vs_stage1_comparison.csv` | Stage 0 vs Stage 1 delta table |
| `plots/confusion_matrix_qchat.png` | Confusion matrix heatmap (best model, test set) |
| `plots/calibration_qchat.png` | XGBoost reliability diagram + probability distribution |
| `plots/shap_summary_qchat.png` | SHAP beeswarm plot (full dataset, 12 features) |
| `plots/shap_bar_qchat.png` | SHAP mean absolute importance bar chart |
| `plots/stage0_vs_stage1_comparison.png` | Side-by-side metric bar chart |
| `models/xgboost_qchat_stage1.pkl` | Final XGBoost trained on all 7530 rows (12 features) |
| `models/qchat_feature_columns.pkl` | Ordered 12-feature column list for inference |
| `data/qchat_probabilities_stage1.csv` | P(ASD) predictions for every row |

### Key Change vs Stage 0

| | Stage 0 | Stage 1 |
|---|---|---|
| Features | 15 (A1–A10 + Sex + Family + Who×3) | 12 (A1–A10 + Sex + Family) |
| Dropped | — | `Who_completed_the_test_*` (3 cols) |
| SHAP API | `shap.TreeExplainer` (**FAILED**) | `shap.Explainer` (XGBoost 2.x compatible) |
| Model file | `xgboost_qchat_combined.pkl` | `xgboost_qchat_stage1.pkl` |